In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Generateur de donnees synthetiques — projet scoring comportemental "Afriland-like". VERSION 3.

NOUVEAUTE v3 : BRUITEUR DE LIBELLES
    Chaque libelle affiche est "sali" au tirage pour imiter un vrai releve : abreviations
    (PAIEMENT->PMT), troncatures (largeur de champ), casse anarchique, separateurs mangles,
    codes d'agence/terminal (AG DLA, TPE1234), references (REF982331), numeros masques
    (69****12), fragments de date (05/03), accents perdus, coquilles. gt_categorie et
    gt_mensonge restent la verite : le texte est sale/trompeur, la verite est connue.

MECANISMES CONSERVES
    (1) Redondance vs incremental ; (2) Biais d'etiquette (secteur informel sur-signale) ;
    (3) Difficulte linguistique (Camfranglais) ; (4) Mensonge sur le libelle (paris/dash).

REGLE : colonnes gt_ = verite-terrain, AUDIT UNIQUEMENT, a RETIRER avant modelisation.
Dependances : numpy, pandas. Deterministe (SEED).
Usage : python generateur_donnees_synthetiques_afriland.py [--n 2000] [--out .]
"""

import argparse
import unicodedata
import numpy as np
import pandas as pd

In [2]:
# ============================ CONFIGURATION ============================
SEED = 42
N_CLIENTS_DEFAUT = 2000
WINDOW_JOURS = 90
TAUX_DEFAUT_CIBLE = 0.14
BIAIS_INFORMEL_FLIP = 0.06
CATS_DECEPTIBLES = ["PARI", "DASH"]
BRUIT = 1.0            # intensite du bruiteur de libelles (0 = propre, 1 = tres sale)

In [3]:
REGIONS = ["Centre", "Littoral", "Ouest", "Nord", "Sud-Ouest", "Sud",
           "Extrême-Nord", "Nord-Ouest", "Adamaoua", "Est"]

LIBELLES = {
    "SALAIRE":   {"std": ["VIREMENT SALAIRE", "PAIE MENSUELLE", "SALAIRE", "VIRT SAL EMPLOYEUR", "VIREMENT", "PAIE"],
                  "signe": +1, "montant": (20000, 1000000)},
    "ELEC":      {"std": ["PAIEMENT ENEO", "ENEO PREPAID", "FACTURE ENEO", "ENEO", "Courant", "Electricité"],
                  "signe": -1, "montant": (500, 50000)},
    "EAU":       {"std": ["FACTURE CAMWATER", "PAIEMENT CAMWATER", "Eau", "Water", "Camwater"],
                  "signe": -1, "montant": (2000, 15000)},
    "LOYER":     {"std": ["LOYER", "PAIEMENT LOYER BAILLEUR", "LOYER MENSUEL", "Maison"],
                  "signe": -1, "montant": (10000, 1500000)},
    "MOMO":      {"std": ["RETRAIT OM", "DEPOT MOMO", "TRANSFERT MTN MOMO", "ORANGE MONEY ENVOI", "Dépot", "Retrait", "Envoi"],
                  "camf": ["OM CASH POUR MOLA", "MOMO SEND SHARP SHARP", "OM RETRAIT NGA", "MTN CASH DEY",
                           "ENVOIE MOMO AU BOSS", "OM POUR LE MBOM", "CASH OUT OM"],
                  "signe": 0, "montant": (1000, 100000)},
    "DASH":      {"std": ["REMBOURSEMENT PRET", "PRET FAMILLE", "AVANCE AMI"],
                  "camf": ["DASH POUR REPO", "PRET DASH VITE", "REMBOURSE DASH MOLA", "DASH SAUVE MOI",
                           "DASH DU FRERE", "PETIT DASH SHARP", "DASH AVANT SALAIRE"],
                  "signe": 0, "montant": (5000, 80000)},
    "NJANGUI":   {"std": ["COTISATION TONTINE", "NJANGUI", "COTIS TONTINE"],
                  "camf": ["NJANGUI QUARTIER", "TONTINE LES MBOM", "NJANGUI DU MOIS", "COTIS NJANGUI",
                           "NJANGUI DES SISTAS", "MAIN NJANGUI"],
                  "signe": -1, "montant": (5000, 50000)},
    "CHOP":      {"std": ["ACHAT MARCHE", "PROVISION ALIMENT", "COURSES MENAGE"],
                  "camf": ["CHOP MARCHE", "PROVISION CHOP", "CHOP DON FINISH", "CHOP DU JOUR",
                           "MARCHE CHOP MAMA", "ON A CHOP"],
                  "signe": -1, "montant": (1000, 30000)},
    "TRANSPORT": {"std": ["TAXI", "CARBURANT", "MOTO TAXI"],
                  "camf": ["BENSKIN", "MOTO SHARP", "TAXI FORT", "DEPOT BENSKIN", "CLANDO"],
                  "signe": -1, "montant": (300, 5000)},
    "PARI":      {"std": ["MISE 1XBET", "PARI FOOT", "MISE MELBET", "PMUC PARI", "COUPON BET", "DEPOT PMUC"],
                  "camf": ["BET LES MBOM", "PARI POUR GAGNER SHARP", "MISE MOTO BET", "BET SHARP SHARP"],
                  "signe": -1, "montant": (500, 40000)},
    "AGIOS":     {"std": ["FRAIS DECOUVERT", "AGIOS", "FRAIS REJET PRELEVEMENT", "COMMISSION DECOUVERT"],
                  "signe": -1, "montant": (500, 15000)},
}
CATS = list(LIBELLES.keys())

MASQUE = ["ACHAT DIVERS", "TRANSFERT FAMILLE", "PROVISION MAISON", "DEPENSE PERSO",
          "ACHAT BOUTIQUE", "MOMO TRANSFERT", "ACHAT MARCHE"]

# ---- ressources du bruiteur ----
ABBR = {
    "PAIEMENT": ["PMT", "PAIMT", "PAIE"], "VIREMENT": ["VIR", "VIRT", "VRT"],
    "TRANSFERT": ["TRF", "TRSF", "TRANS"], "FACTURE": ["FACT", "FAC", "FCT"],
    "RETRAIT": ["RET", "RETR", "RTR"], "REMBOURSEMENT": ["RBST", "REMB", "RMB"],
    "COTISATION": ["COTIS", "COT"], "SALAIRE": ["SAL", "SALR"], "PROVISION": ["PROV", "PRV"],
    "ACHAT": ["ACH", "ACHT"], "MENSUELLE": ["MENS", "MSL"], "ORANGE": ["OR", "ORG"],
    "MONEY": ["MNY", "MON"], "ELECTRICITE": ["ELEC", "ELC"], "CARBURANT": ["CARB", "CARBU"],
    "DEPOT": ["DEP", "DPT"], "COMMISSION": ["COMM", "COM"], "FRAIS": ["FRS", "FR"],
    "PRELEVEMENT": ["PRLV", "PREL"], "FAMILLE": ["FAM", "FML"], "DEPENSE": ["DEP", "DPNS"],
    "MARCHE": ["MRCH", "MCHE"], "BAILLEUR": ["BAIL", "BLR"], "DECOUVERT": ["DECOUV", "DCV"],
}
VILLES = ["DLA", "YDE", "BAF", "BUE", "GAR", "MAR", "BER", "EBW", "KRI", "NGD"]
SEPS = [" ", "  ", ".", "/", "-", "", "_", "*"]

In [4]:
def strip_accents(s):
    return unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode("ascii")

In [5]:
def bruiter(base, cat, camf_intensity, rng):
    """Salit un libelle au maximum pour imiter un releve de terrain ultra-brut."""
    if rng.uniform() > BRUIT:
        return base

    mots = base.split()

    # 1. Abreviations massives (ex: PAIEMENT -> PMT)
    mots = [rng.choice(ABBR[m.upper()]) if (m.upper() in ABBR and rng.uniform() < 0.85) else m for m in mots]
    s = " ".join(mots)

    # 2. Perte systematique des accents
    s = strip_accents(s)

    # 3. Injection de prefixes canal agressifs
    if rng.uniform() < 0.70:
        pref = rng.choice(["TRF", "VIR", "PMT", "RET GAB", "ACH TPE", "PAIMT", "OPR", "DEB", "ENCASXT"])
        s = pref + " " + s

    # 4. Casse anarchique au caractere pres (Ex: pMt_vIReMeNt)
    s = "".join(c.upper() if rng.uniform() < 0.5 else c.lower() for c in s)

    # 5. Separateurs manges et colles (remplace les espaces par du bruit)
    if rng.uniform() < 0.75:
        s = "".join(rng.choice(SEPS) if c == " " else c for c in s)

    # 6. Suffixes techniques obligatoires (Agences, TPE, terminaux, dates)
    suf = []
    if rng.uniform() < 0.70:
        suf.append("REF" + str(rng.integers(10000, 9999999)))
    if rng.uniform() < 0.50:
        suf.append("AG" + rng.choice(VILLES))
    if rng.uniform() < 0.50:
        suf.append("TPE" + str(rng.integers(1000, 99999)))
    if cat in ("MOMO", "DASH", "PARI", "MASQUE") and rng.uniform() < 0.65:
        suf.append("6" + str(rng.integers(50, 99)) + "****" + str(rng.integers(10, 99)))
    if rng.uniform() < 0.40:
        suf.append(f"{rng.integers(1,28):02d}/{rng.integers(1,12):02d}")

    if suf:
        # Parfois on colle le suffixe directement sans espace
        separateur_suf = rng.choice([" ", "", "/", "*"])
        s = s + separateur_suf + separateur_suf.join(suf)

    # 7. Coquilles agressives (lettres doublees, mangees ou parasitees)
    if len(s) > 4 and rng.uniform() < 0.40:
        for _ in range(rng.integers(1, 3)):  # jusqu a 2 coquilles par libelle
            i = int(rng.integers(0, len(s)))
            r_coquille = rng.uniform()
            if r_coquille < 0.33:
                s = s[:i] + (s[i] * 2 if i < len(s) else "") + s[i+1:]  # lettre doublee
            elif r_coquille < 0.66:
                s = s[:i] + s[i+1:]  # lettre mangee
            else:
                s = s[:i] + rng.choice(["X", "Z", "9", "*", "$"]) + s[i+1:]  # parasite

    # 8. Troncature brutale facon largeur de champ fixe
    if rng.uniform() < 0.45:
        s = s[:int(rng.integers(8, 25))].rstrip()

    return s if len(s.strip()) >= 3 else base

In [6]:
def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))

In [7]:
def solve_intercept(z, cible, lo=-10, hi=10, iters=60):
    for _ in range(iters):
        mid = 0.5 * (lo + hi)
        if sigmoid(mid - z).mean() > cible:
            hi = mid
        else:
            lo = mid
    return 0.5 * (lo + hi)

In [8]:
def generer(n, rng):
    sexe = rng.choice(["F", "M"], size=n)
    zone = rng.choice(["urbaine", "semi_urbaine"], size=n, p=[0.6, 0.4])
    secteur = rng.choice(["formel", "informel"], size=n, p=[0.45, 0.55])
    region = rng.choice(REGIONS, size=n)
    age = rng.integers(18, 36, size=n)
    anciennete = rng.integers(1, 96, size=n).astype(float)

    camf = sigmoid(0.9 * (secteur == "informel") + 0.6 * (zone == "semi_urbaine")
                   + 0.4 * ((30 - age) / 12.0) + rng.normal(0, 0.5, n) - 0.3)
    income_stab = rng.normal(0.6 * (secteur == "formel"), 1.0, n)
    distress = rng.normal(0.5 * (secteur == "informel") + 0.2 * camf, 1.0, n)
    hidden_support = rng.normal(0.3 * (zone == "semi_urbaine"), 1.0, n)
    p_mensonge = sigmoid(-1.3 + 0.9 * distress + 0.3 * (secteur == "informel") + rng.normal(0, 0.6, n))

    C_star = (0.9 * income_stab - 1.0 * distress + 0.5 * hidden_support
              + 0.010 * anciennete + rng.normal(0, 0.6, n))
    C_star = (C_star - C_star.mean()) / C_star.std()
    b = solve_intercept(1.4 * C_star, TAUX_DEFAUT_CIBLE)
    defaut_true = (rng.uniform(size=n) < sigmoid(b - 1.4 * C_star)).astype(int)
    defaut_obs = defaut_true.copy()
    flips = (secteur == "informel") & (defaut_true == 0) & (rng.uniform(size=n) < BIAIS_INFORMEL_FLIP)
    defaut_obs[flips] = 1

    revenu_reel = np.clip(120000 + 60000 * income_stab, 30000, None)
    revenu_declare = np.round((revenu_reel * (1 + 0.15 * np.clip(distress, 0, None))) / 1000) * 1000

    return pd.DataFrame({
        "client_id": np.arange(1, n + 1),
        "age": age, "sexe": sexe, "zone": zone, "secteur": secteur, "region": region,
        "revenu_declare": revenu_declare, "anciennete_mois": anciennete.astype(int),
        "defaut_90j": defaut_obs,
        "gt_C_star": np.round(C_star, 4), "gt_income_stab": np.round(income_stab, 4),
        "gt_distress": np.round(distress, 4), "gt_hidden_support": np.round(hidden_support, 4),
        "gt_camf_intensity": np.round(camf, 4), "gt_p_mensonge": np.round(p_mensonge, 4),
        "gt_defaut_true": defaut_true, "gt_revenu_reel": np.round(revenu_reel / 1000) * 1000,
    })

In [9]:
def poids_categories(row):
    d, s, h = row.gt_distress, row.gt_income_stab, row.gt_hidden_support
    est_M = (row.sexe == "M"); jeune = max((28 - row.age) / 10.0, 0)
    w = {"SALAIRE": 0.6 + 0.5 * max(s, 0), "ELEC": 0.5 + 0.4 * max(s, 0), "EAU": 0.4 + 0.3 * max(s, 0),
         "LOYER": 0.5, "MOMO": 1.2 + 0.6 * (row.secteur == "informel"), "DASH": 0.3 + 0.9 * max(d, 0),
         "NJANGUI": 0.3 + 0.9 * max(h, 0), "CHOP": 1.0 + 0.4 * (row.secteur == "informel"),
         "TRANSPORT": 0.8, "PARI": 0.2 + 0.7 * max(d, 0) + 0.3 * est_M + 0.2 * jeune,
         "AGIOS": 0.1 + 0.7 * max(d, 0)}
    v = np.array([w[c] for c in CATS], dtype=float)
    return v / v.sum()

In [10]:
def generer_transactions(clients, rng):
    lignes = []
    for row in clients.itertuples(index=False):
        activite = 30 + 25 * max(row.gt_income_stab, -1) + 15 * (row.secteur == "informel")
        n_tx = int(np.clip(rng.poisson(max(activite, 8)), 8, 160))
        cats = rng.choice(CATS, size=n_tx, p=poids_categories(row))
        jours = np.sort(rng.integers(0, WINDOW_JOURS, size=n_tx))
        for cat, j in zip(cats, jours):
            spec = LIBELLES[cat]; ment = 0
            if cat in CATS_DECEPTIBLES and rng.uniform() < row.gt_p_mensonge:
                base = rng.choice(MASQUE); ment = 1
                lib = bruiter(base, "MASQUE", row.gt_camf_intensity, rng)
            else:
                if "camf" in spec and rng.uniform() < row.gt_camf_intensity:
                    base = rng.choice(spec["camf"])
                else:
                    base = rng.choice(spec["std"])
                lib = bruiter(base, cat, row.gt_camf_intensity, rng)
            lo, hi = spec["montant"]; mnt = float(rng.integers(lo, hi))
            signe = spec["signe"] or rng.choice([+1, -1])
            date = np.datetime64("2024-01-01") + np.timedelta64(int(j), "D")
            lignes.append((row.client_id, str(date), round(signe * mnt, 2), lib, cat, ment))
    return pd.DataFrame(lignes, columns=["client_id", "date", "montant", "libelle",
                                         "gt_categorie", "gt_mensonge"])

In [11]:
def controle_qualite(clients, tx):
    print(f"Clients : {len(clients)} | Transactions : {len(tx)}")
    print(f"Taux defaut observe {clients.defaut_90j.mean():.3f} | vrai {clients.gt_defaut_true.mean():.3f}")
    dec = tx[tx.gt_categorie.isin(CATS_DECEPTIBLES)]
    print(f"\n(4) MENSONGE PARI/DASH : taux global {dec.gt_mensonge.mean():.3f} | "
          f"par defaut vrai {dec.merge(clients[['client_id','gt_defaut_true']],on='client_id').groupby('gt_defaut_true').gt_mensonge.mean().round(3).to_dict()}")
    print("\nEXEMPLES de libelles bruites (libelle | vraie categorie | mensonge) :")
    ech = tx.sample(16, random_state=0)
    for r in ech.itertuples():
        print(f"  {r.libelle[:38]:<38} | {r.gt_categorie:<9} | {r.gt_mensonge}")

In [12]:
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--n", type=int, default=N_CLIENTS_DEFAUT)
    ap.add_argument("--out", type=str, default=".")
    
    # CORRECTION : On passe une liste vide pour éviter que Jupyter n'injecte ses paramètres
    args = ap.parse_args(args=[]) 
    
    rng = np.random.default_rng(SEED)
    clients = generer(args.n, rng)
    tx = generer_transactions(clients, rng)
    
    c_path = f"{args.out.rstrip('/')}/clients_synth.csv"
    t_path = f"{args.out.rstrip('/')}/transactions_synth.csv"
    
    clients.to_csv(c_path, index=False)
    tx.to_csv(t_path, index=False)
    
    controle_qualite(clients, tx)
    print(f"\nFichiers ecrits : {c_path} | {t_path}")

if __name__ == "__main__":
    main()

Clients : 2000 | Transactions : 91666
Taux defaut observe 0.159 | vrai 0.134

(4) MENSONGE PARI/DASH : taux global 0.436 | par defaut vrai {0: 0.418, 1: 0.543}

EXEMPLES de libelles bruites (libelle | vraie categorie | mensonge) :
  PMT  Pret.dASh_VitE*REF1               | DASH      | 0
  PmT vIrE                               | SALAIRE   | 0
  Fr.rejeT pRElREF52                     | AGIOS     | 0
  PaiMT-bet  lES_mBoM/REF1332677/AGBUE/T | PARI      | 0
  enCASXT eNvOIREF2874978AGEBWTPE1384069 | MOMO      | 0
  dEbCHOP dU JoUr                        | CHOP      | 0
  MISe  mElbEt REF5860                   | PARI      | 0
  AvaNcE AMI RE                          | DASH      | 0
  pMt dPt.mOMo REF8774847 AGGAR          | MOMO      | 0
  VIR_PeTi$DASH S                        | DASH      | 0
  OPr AcHT mChE/REF13Z1413/AGDDLA/668*** | DASH      | 1
  pAIMt.prv aLImeNTREF2470558            | CHOP      | 0
  tf FMl REEF9665146 AGDLA 670****13     | DASH      | 1
  ReT gab MIsE melbet/REF448